<a href="https://colab.research.google.com/github/MohamedTareqFarouq/Pest-Classification/blob/main/MobileNetV2_72Class_Drive_Resume_waveletBranchWithOriginalImage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MobileNetV2 baseline on the 72-class pest dataset — Drive checkpoints

Plain MobileNetV2 baseline (no wavelet loss, no Sobel loss, no decoder) — Drive checkpoints with resume.

## 1. Install dependencies

In [ ]:
!pip -q install gdown split-folders scikit-learn

## 2. Imports and experiment settings

In [ ]:
import random
import os
import time
import json
import hashlib
import zipfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.auto import tqdm

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
RESUME = True
PATIENCE = 7
NUM_WORKERS = 0

BACKBONE_LR = 1e-4
HEAD_LR = 1e-3

RUN_NAME = "mobilenetv2_72class"
# Keep RUN_NAME unchanged when continuing the same experiment.
# Optional: path to an older weights-only checkpoint in Drive.
LEGACY_CHECKPOINT = ""

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
print("Device:", device)

Device: cuda


## 3. Mount Google Drive and checkpoint paths

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

RUN_DIR = Path("/content/drive/MyDrive/Pest_Project") / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
SAVE_PATH = RUN_DIR / "best.pth"
LATEST_PATH = RUN_DIR / "latest.pth"
PREVIOUS_PATH = RUN_DIR / "previous.pth"

if not RESUME and any(p.exists() for p in [SAVE_PATH, LATEST_PATH, PREVIOUS_PATH]):
    raise RuntimeError("This run already has checkpoints. Enable RESUME or use a new RUN_NAME.")

print("Checkpoints will be saved to:", RUN_DIR)

Mounted at /content/drive
Checkpoints will be saved to: /content/drive/MyDrive/Pest_Project/mobilenetv2_72class


## 4. Prepare the dataset (72-class)

In [ ]:
import gdown
import splitfolders

RAW_DIR = Path("/content/Cleaned_OurDataSet")
SPLIT_DIR = Path("/content/dataset_split")
ZIP_PATH = Path("/content/Cleaned_OurDataSet.zip")

split_exists = all(
    (SPLIT_DIR / name).is_dir()
    for name in ["train", "val", "test"]
)

if split_exists:
    print("Using the existing dataset split.")

else:
    if SPLIT_DIR.exists():
        raise RuntimeError(
            "dataset_split exists but is incomplete. "
            "Use a fresh output folder or fix that split first."
        )

    if not RAW_DIR.exists():
        if not ZIP_PATH.exists():
            gdown.download(
                id="1I5Of433HE2etWS49klL0Zwp1DHd122wa",
                output=str(ZIP_PATH),
                quiet=False,
            )

        with zipfile.ZipFile(ZIP_PATH, "r") as archive:
            archive.extractall("/content")

    if not RAW_DIR.is_dir():
        raise FileNotFoundError(
            f"Dataset not found at {RAW_DIR}. Check the extracted folder name."
        )

    splitfolders.ratio(
        str(RAW_DIR),
        output=str(SPLIT_DIR),
        seed=SEED,
        ratio=(0.8, 0.1, 0.1),
        move=False,
    )

    print("Created an 80/10/10 split.")

DATA_ROOT = SPLIT_DIR  # used for the dataset manifest

Downloading...
From (original): https://drive.google.com/uc?id=1I5Of433HE2etWS49klL0Zwp1DHd122wa
From (redirected): https://drive.google.com/uc?id=1I5Of433HE2etWS49klL0Zwp1DHd122wa&confirm=t&uuid=d9da5965-9c24-4122-8b3e-ac7d4c27e453
To: /content/Cleaned_OurDataSet.zip
100%|██████████| 2.42G/2.42G [00:51<00:00, 47.0MB/s]
Copying files: 118664 files [00:35, 3308.25 files/s]

Created an 80/10/10 split.


## 5. Images, loaders, and split validation

In [ ]:
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.8, 1.0),
    ),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(
    SPLIT_DIR / "train",
    transform=train_transforms,
)

val_dataset = datasets.ImageFolder(
    SPLIT_DIR / "val",
    transform=eval_transforms,
)

test_dataset = datasets.ImageFolder(
    SPLIT_DIR / "test",
    transform=eval_transforms,
)

assert (
    train_dataset.class_to_idx
    == val_dataset.class_to_idx
    == test_dataset.class_to_idx
), "The train, validation, and test class mappings must match."

class_names = train_dataset.classes
NUM_CLASSES = len(class_names)


def make_loaders():
    # Reset the shuffle generator for each experiment.
    generator = torch.Generator().manual_seed(SEED)

    common = {
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "pin_memory": device.type == "cuda",
    }

    train_loader = DataLoader(
        train_dataset,
        shuffle=True,
        generator=generator,
        **common,
    )

    val_loader = DataLoader(
        val_dataset,
        shuffle=False,
        **common,
    )

    test_loader = DataLoader(
        test_dataset,
        shuffle=False,
        **common,
    )

    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = make_loaders()

print("Classes:", class_names)
print("Number of classes:", NUM_CLASSES)
print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))
# Record split membership and sizes so a different dataset cannot silently resume.
manifest = {}
for split, dataset in [("train", train_dataset), ("val", val_dataset), ("test", test_dataset)]:
    manifest[split] = [
        [Path(path).relative_to(DATA_ROOT).as_posix(), label, Path(path).stat().st_size]
        for path, label in dataset.samples
    ]
dataset_signature = hashlib.sha256(
    json.dumps(manifest, sort_keys=True).encode()
).hexdigest()
manifest_path = RUN_DIR / "dataset_manifest.json"
if manifest_path.exists():
    if json.loads(manifest_path.read_text()) != manifest:
        raise RuntimeError("Dataset split changed. Restore the original split or use a new RUN_NAME.")
else:
    manifest_path.write_text(json.dumps(manifest, indent=2))

Classes: ['Acalymma_vittatum', 'Achatina_fulica', 'Alticini', 'Ampelophaga', 'Anasa_tristis', 'Aphids', 'Armyworm_butterfly', 'Armyworm_caterpillar', 'Aulacophora_similis', 'Beet spot flies_butterfly', 'Beet spot flies_caterpillar', 'Cicadella viridis', 'Cicadellidae', 'Dermaptera', 'Icerya purchasi Maskell', 'Leptinotarsa_decemlineata', 'Locustoidea', 'Lycorma delicatula', 'Mantodea', 'Miridae', 'Potosiabre vitarsis', 'Prodenia litura_butterfly', 'Prodenia litura_caterpillar', 'Rhytidodera bowrinii white', 'Salurnis marginella Guerr', 'Thrips', 'Xylotrechus', 'alfalfa plant bug_butterfly', 'alfalfa plant bug_caterpillar', 'army worm_butterfly', 'army worm_caterpillar', 'asiatic rice borer_butterfly', 'asiatic rice borer_caterpillar', 'beet army worm_butterfly', 'beet army worm_caterpillar', 'beetle', 'black cutworm_butterfly', 'black cutworm_caterpillar', 'blister beetle', 'bollworm_butterfly', 'bollworm_caterpillar', 'cabbage army worm', 'cerodonta denticornis_butterfly', 'cerodonta 

## 6. MobileNetV2 model

In [ ]:
class MobileNetV2Classifier(nn.Module):
    """Plain MobileNetV2 baseline: pretrained backbone + dropout/linear head.
    No decoder, no wavelet loss, no Sobel loss."""

    def __init__(self, num_classes, pretrained=True):
        super().__init__()

        weights = (
            models.MobileNet_V2_Weights.DEFAULT
            if pretrained else None
        )

        mobilenet = models.mobilenet_v2(weights=weights)

        self.features = mobilenet.features

        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(1280, num_classes),
        )

        self.register_buffer(
            "mean",
            torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1),
        )
        self.register_buffer(
            "std",
            torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1),
        )

    def forward(self, images):
        normalized = (images - self.mean) / self.std
        features = self.features(normalized)
        pooled = F.adaptive_avg_pool2d(features, 1).flatten(1)
        return self.classifier(pooled)

In [ ]:
class HaarDWT2D(nn.Module):
    """Computes 2D Haar Discrete Wavelet Transform on a batch of images using 2D Conv."""
    def __init__(self, in_channels=3):
        super().__init__()
        self.in_channels = in_channels

        # Haar filters: Low-Low (LL), Low-High (LH), High-Low (HL), High-High (HH)
        ll = torch.tensor([[0.5, 0.5], [0.5, 0.5]])
        lh = torch.tensor([[0.5, 0.5], [-0.5, -0.5]])
        hl = torch.tensor([[0.5, -0.5], [0.5, -0.5]])
        hh = torch.tensor([[0.5, -0.5], [-0.5, 0.5]])

        # Stack filters: shape [4, 1, 2, 2]
        filters = torch.stack([ll, lh, hl, hh], dim=0).unsqueeze(1)

        # Repeat filters for each input channel (grouped convolution)
        # Output shape will be [batch, in_channels * 4, H/2, W/2]
        weight = torch.zeros(in_channels * 4, 1, 2, 2)
        for i in range(in_channels):
            weight[i*4 : (i+1)*4, 0, :, :] = filters

        self.register_buffer("weight", weight)

    def forward(self, x):
        # Grouped convolution with stride 2 for downsampling
        out = F.conv2d(x, self.weight, bias=None, stride=2, padding=0, groups=self.in_channels)
        return out

In [ ]:
class MobileNetV2WaveletClassifier(nn.Module):
    """MobileNetV2 with early Haar Wavelet integration in the backbone."""
    def __init__(self, num_classes, pretrained=True):
        super().__init__()

        weights = models.MobileNet_V2_Weights.DEFAULT if pretrained else None
        mobilenet = models.mobilenet_v2(weights=weights)

        # Extract the first ConvNormActivation block of MobileNetV2
        # Input: 3 channels (224x224) -> Output: 32 channels (112x112)
        self.first_block = mobilenet.features[0]

        # Remaining blocks of MobileNetV2 features
        self.remaining_features = nn.Sequential(*list(mobilenet.features[1:]))

        # 2D Haar DWT: Input: 3 channels (224x224) -> Output: 12 channels (112x112)
        self.dwt = HaarDWT2D(in_channels=3)

        # Concatenated channels: 32 (from first_block) + 12 (from DWT) = 44 channels
        # Project 44 channels back down to 32 channels to match the input of the second block
        self.projector = nn.Sequential(
            nn.Conv2d(44, 32, kernel_size=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True)
        )

        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(1280, num_classes),
        )

        self.register_buffer(
            "mean",
            torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1),
        )
        self.register_buffer(
            "std",
            torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1),
        )

    def forward(self, images):
        normalized = (images - self.mean) / self.std

        # 1. Pass image through the first layer of MobileNetV2 backbone (output: [B, 32, 112, 112])
        f_first = self.first_block(normalized)

        # 2. Extract Haar DWT subbands from raw normalized image (output: [B, 12, 112, 112])
        f_dwt = self.dwt(normalized)

        # 3. Concatenate along channel dimension (output: [B, 44, 112, 112])
        f_concat = torch.cat([f_first, f_dwt], dim=1)

        # 4. Project back down to 32 channels (output: [B, 32, 112, 112])
        f_projected = self.projector(f_concat)

        # 5. Feed into rest of MobileNetV2 backbone and classifier
        features = self.remaining_features(f_projected)
        pooled = F.adaptive_avg_pool2d(features, 1).flatten(1)
        return self.classifier(pooled)

In [ ]:
# Verification test to ensure shapes align correctly
test_input = torch.randn(1, 3, 224, 224)
test_model = MobileNetV2WaveletClassifier(num_classes=72, pretrained=False)
test_output = test_model(test_input)
print("Input Shape:", test_input.shape)
print("Output Shape:", test_output.shape)
print("Network verification complete! Your backbone is integrated with 2D Haar Wavelets.")

## 7. Initialize model and optimizer

In [ ]:
set_seed(SEED)
train_loader, val_loader, test_loader = make_loaders()

has_saved_weights = any(p.exists() for p in [LATEST_PATH, PREVIOUS_PATH, SAVE_PATH]) or bool(LEGACY_CHECKPOINT)
model = MobileNetV2Classifier(
    NUM_CLASSES, pretrained=not has_saved_weights
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW([
    {
        "params": model.features.parameters(),
        "lr": BACKBONE_LR,
    },
    {
        "params": model.classifier.parameters(),
        "lr": HEAD_LR,
    },
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)

print("Model ready: MobileNetV2 baseline,", NUM_CLASSES, "classes.")

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 146MB/s]


Model ready: MobileNetV2 baseline, 72 classes.


## 8. Training and validation functions

In [ ]:
def train_one_epoch(model, loader):
    model.train()

    total_loss = 0.0
    correct = 0
    count = 0

    for images, labels in tqdm(loader, desc="Training", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        batch_size = labels.size(0)
        count += batch_size
        total_loss += loss.item() * batch_size
        correct += (logits.argmax(dim=1) == labels).sum().item()

    return {
        "loss": total_loss / count,
        "accuracy": correct / count,
    }


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    total_loss = 0.0
    correct = 0
    count = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        batch_size = labels.size(0)
        count += batch_size
        total_loss += loss.item() * batch_size

        correct += (logits.argmax(dim=1) == labels).sum().item()

    return {
        "loss": total_loss / count,
        "accuracy": correct / count,
    }

## 9. Restore the saved training state

In [ ]:
# Checkpoints use tensors and basic Python values: safe for weights_only=True.
def atomic_save(payload, path):
    temporary = path.with_name(path.name + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def capture_rng():
    np_state = np.random.get_state()
    return {
        "python": random.getstate(),
        "numpy": [np_state[0], np_state[1].tolist(), np_state[2], np_state[3], np_state[4]],
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
        "loader": train_loader.generator.get_state(),
    }


def restore_rng(state):
    random.setstate(state["python"])
    name, values, position, has_gauss, cached = state["numpy"]
    np.random.set_state((name, np.array(values, dtype=np.uint32), position, has_gauss, cached))
    torch.set_rng_state(state["torch"].cpu())
    train_loader.generator.set_state(state["loader"].cpu())
    if torch.cuda.is_available() and len(state["cuda"]) == torch.cuda.device_count():
        torch.cuda.set_rng_state_all([value.cpu() for value in state["cuda"]])


def validate_checkpoint(checkpoint):
    if checkpoint["class_names"] != class_names:
        raise RuntimeError("Checkpoint classes do not match this dataset.")
    if checkpoint["image_size"] != IMAGE_SIZE:
        raise RuntimeError("Checkpoint settings do not match this experiment.")
    if "dataset_signature" in checkpoint and checkpoint["dataset_signature"] != dataset_signature:
        raise RuntimeError("Checkpoint dataset split does not match.")


history = []
start_epoch = 1
best_accuracy = -1.0
best_val_loss = float("inf")
epochs_without_improvement = 0

# Prefer the latest complete epoch, then the previous complete epoch.
resume_checkpoint = None
resume_path = None
if RESUME:
    for path in [LATEST_PATH, PREVIOUS_PATH, SAVE_PATH]:
        if path.exists():
            try:
                resume_checkpoint = torch.load(path, map_location="cpu", weights_only=True)
            except (OSError, RuntimeError, EOFError) as error:
                print("Could not read", path.name, ":", error)
                continue
            validate_checkpoint(resume_checkpoint)
            resume_path = path
            break
    if resume_checkpoint is None and any(p.exists() for p in [LATEST_PATH, PREVIOUS_PATH, SAVE_PATH]):
        raise RuntimeError("Saved checkpoints could not be read. Restore a working backup first.")
    if resume_checkpoint is None and LEGACY_CHECKPOINT:
        resume_path = Path(LEGACY_CHECKPOINT)
        resume_checkpoint = torch.load(resume_path, map_location="cpu", weights_only=True)
        validate_checkpoint(resume_checkpoint)

if resume_checkpoint is not None:
    checkpoint = resume_checkpoint
    model.load_state_dict(checkpoint["model_state"])
    start_epoch = checkpoint["epoch"] + 1
    best_accuracy = checkpoint.get("best_accuracy", checkpoint["val_accuracy"])
    best_val_loss = checkpoint.get("best_val_loss", checkpoint["val_loss"])
    history = checkpoint.get("history", [])
    epochs_without_improvement = checkpoint.get("epochs_without_improvement", 0)

    if "optimizer_state" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        scheduler.load_state_dict(checkpoint["scheduler_state"])
        restore_rng(checkpoint["rng_state"])
        print("Restored model, optimizer, scheduler, history, and random states.")
    else:
        print("Old checkpoint: restored weights; optimizer and scheduler start fresh.")
        # Preserve this older best model before training continues.
        if not SAVE_PATH.exists():
            atomic_save(checkpoint, SAVE_PATH)

    print("Restored:", resume_path)
    print("Next epoch:", start_epoch)
else:
    print("Starting a new experiment.")

Starting a new experiment.


## 10. Train and save to Drive

In [ ]:
if epochs_without_improvement >= PATIENCE:
    print("This run already reached early stopping. Continue to evaluation.")

for epoch in range(start_epoch, EPOCHS + 1):
    if epochs_without_improvement >= PATIENCE:
        break
    epoch_start = time.time()
    train_metrics = train_one_epoch(model, train_loader)
    val_metrics = evaluate(model, val_loader)

    scheduler.step(val_metrics["loss"])

    history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
    })

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train loss: {train_metrics['loss']:.4f} | "
        f"Val loss: {val_metrics['loss']:.4f} | "
        f"Train accuracy: {train_metrics['accuracy']:.2%} | "
        f"Val accuracy: {val_metrics['accuracy']:.2%}"
    )

    improved = (
        val_metrics["accuracy"] > best_accuracy
        or (
            val_metrics["accuracy"] == best_accuracy
            and val_metrics["loss"] < best_val_loss
        )
    )

    if improved:
        best_accuracy = val_metrics["accuracy"]
        best_val_loss = val_metrics["loss"]
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    payload = {
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "class_names": class_names,
        "image_size": IMAGE_SIZE,
        "dataset_signature": dataset_signature,
        "epoch": epoch,
        "val_accuracy": val_metrics["accuracy"],
        "val_loss": val_metrics["loss"],
        "best_accuracy": best_accuracy,
        "best_val_loss": best_val_loss,
        "epochs_without_improvement": epochs_without_improvement,
        "history": history,
        "rng_state": capture_rng(),
    }

    if improved:
        atomic_save(payload, SAVE_PATH)
        print("Saved best model to Google Drive.")

    # Keep an additional previous checkpoint during replacement.
    if LATEST_PATH.exists():
        os.replace(LATEST_PATH, PREVIOUS_PATH)
    atomic_save(payload, LATEST_PATH)
    print(f"Saved epoch {epoch} for resume. Time: {(time.time() - epoch_start) / 60:.1f} min")
    start_epoch = epoch + 1

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping.")
        break

print(f"\nBest validation accuracy: {best_accuracy:.2%}")
print("Checkpoint:", SAVE_PATH)

Training:   0%|          | 0/2966 [00:00<?, ?it/s]

## 11. Plot training history

In [ ]:
epochs = [row["epoch"] for row in history]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(
    epochs,
    [row["train_accuracy"] for row in history],
    label="Train",
)
axes[0].plot(
    epochs,
    [row["val_accuracy"] for row in history],
    label="Validation",
)
axes[0].set_title("Classification accuracy")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(
    epochs,
    [row["train_loss"] for row in history],
    label="Train CE",
)
axes[1].plot(
    epochs,
    [row["val_loss"] for row in history],
    label="Validation CE",
)
axes[1].set_title("Classification loss")
axes[1].legend()

for axis in axes:
    axis.set_xlabel("Epoch")
    axis.grid(alpha=0.3)

plt.tight_layout()
plt.show()
fig.savefig(RUN_DIR / "training_curves.png", bbox_inches="tight")
(RUN_DIR / "history.json").write_text(json.dumps(history, indent=2))

## 12. Load the best model for evaluation

In [ ]:
checkpoint = torch.load(
    SAVE_PATH,
    map_location=device,
    weights_only=True,
)

assert checkpoint["class_names"] == class_names

model.load_state_dict(checkpoint["model_state"])
model.eval()

print("Loaded epoch:", checkpoint["epoch"])
print(
    "Best validation accuracy:",
    f"{checkpoint['val_accuracy']:.2%}",
)

## 13. Final test evaluation

In [ ]:
test_loss_sum = 0.0
test_count = 0

true_labels = []
predicted_labels = []

model.eval()

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        logits = model(images.to(device))
        test_loss_sum += criterion(logits, labels.to(device)).item() * labels.size(0)
        test_count += labels.size(0)
        predictions = logits.argmax(dim=1)

        true_labels.extend(labels.tolist())
        predicted_labels.extend(predictions.cpu().tolist())

test_accuracy = np.mean(np.array(true_labels) == np.array(predicted_labels))
print(f"Test accuracy: {test_accuracy:.2%}")
print(f"Test classification loss: {test_loss_sum / test_count:.4f}")

report = classification_report(
    true_labels,
    predicted_labels,
    labels=list(range(NUM_CLASSES)),
    target_names=class_names,
    digits=4,
    zero_division=0,
)
print(report)

matrix = confusion_matrix(
    true_labels,
    predicted_labels,
    labels=list(range(NUM_CLASSES)),
)

figure_size = min(24, max(8, NUM_CLASSES * 0.45))

fig, axis = plt.subplots(figsize=(figure_size, figure_size))
image = axis.imshow(matrix, cmap="Blues")

tick_size = 6 if NUM_CLASSES <= 80 else 4
axis.set_xticks(range(NUM_CLASSES))
axis.set_yticks(range(NUM_CLASSES))
axis.set_xticklabels(class_names, rotation=90, fontsize=tick_size)
axis.set_yticklabels(class_names, fontsize=tick_size)

axis.set_xlabel("Predicted class")
axis.set_ylabel("True class")
axis.set_title("Test confusion matrix")

fig.colorbar(image, ax=axis)
plt.tight_layout()
plt.show()

(RUN_DIR / "test_report.txt").write_text(
    f"Best model epoch: {checkpoint['epoch']}\nTest accuracy: {test_accuracy:.4%}\n\n" + report
)
fig.savefig(RUN_DIR / "test_confusion_matrix.png", bbox_inches="tight")

## 14. Optional single-image prediction

In [ ]:
from PIL import Image

IMAGE_PATH = "/content/your_pest_image.jpg"

if not Path(IMAGE_PATH).is_file():
    print("Optional: set IMAGE_PATH to a real image, then rerun this cell.")
else:
    image = Image.open(IMAGE_PATH).convert("RGB")
    image_tensor = eval_transforms(image).unsqueeze(0).to(device)

    model.eval()

    with torch.no_grad():
        logits = model(image_tensor)
        probabilities = torch.softmax(logits, dim=1)[0]

    score, predicted_index = probabilities.max(dim=0)

    predicted_class = class_names[predicted_index.item()]

    print("Predicted pest:", predicted_class)
    print(f"Model probability: {score.item():.2%}")

    plt.imshow(image)
    plt.title(predicted_class)
    plt.axis("off")
    plt.show()